# 05 — DGIdb & Open Targets Cross-Check

**Stage 6:** cross-check our EGFR drugs against two independent sources.
- **DGIdb** — drug–gene interactions (do other databases agree our drugs hit EGFR?)
- **Open Targets** — which diseases EGFR is associated with (e.g. lung cancers)

Outputs: `egfr_dgidb_interactions.csv`, `egfr_opentargets_associations.csv`, `egfr_external_crosscheck_summary.csv`

### 1. Test notebook environment

In [1]:
import sys
import time
import re
from pathlib import Path

import requests
import pandas as pd

print("Notebook is working")
print("Python executable:", sys.executable)

Notebook is working
Python executable: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/.venv/bin/python


### 2. Set project folders

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Processed data folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


### 3. Load existing dataset files

In [3]:
recommendations_file = PROCESSED_DIR / "egfr_drug_recommendations.csv"
pubmed_summary_file = PROCESSED_DIR / "egfr_pubmed_summary.csv"
clinical_trials_summary_file = PROCESSED_DIR / "egfr_clinical_trials_summary.csv"
openfda_summary_file = PROCESSED_DIR / "egfr_openfda_summary.csv"

if not recommendations_file.exists():
    raise FileNotFoundError("egfr_drug_recommendations.csv not found. Run Notebook 01 first.")

drug_recommendations_df = pd.read_csv(recommendations_file)
print("Drug recommendations:", len(drug_recommendations_df))

for name, fp in {
    "PubMed summary": pubmed_summary_file,
    "Clinical trials summary": clinical_trials_summary_file,
    "openFDA summary": openfda_summary_file,
}.items():
    print(name, "exists:", fp.exists())

drug_recommendations_df.head()

Drug recommendations: 76
PubMed summary exists: True
Clinical trials summary exists: True
openFDA summary exists: True


,drug_name,molecule_chembl_id,action_type,mechanism_of_action,approval_status,max_phase,target_name
0,PANITUMUMAB,CHEMBL1201827,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
1,CETUXIMAB,CHEMBL1201577,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
2,ERLOTINIB HYDROCHLORIDE,CHEMBL1079742,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
3,GEFITINIB,CHEMBL939,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
4,LAPATINIB DITOSYLATE,CHEMBL1201179,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor


### 4. Set target metadata

In [4]:
target_name = "EGFR"
target_full_name = "Epidermal growth factor receptor"
target_chembl_id = "CHEMBL203"
target_ensembl_id = "ENSG00000146648"  # EGFR Ensembl gene id (Open Targets)

print("Target name:", target_name)
print("Target full name:", target_full_name)
print("Target ChEMBL ID:", target_chembl_id)
print("Target Ensembl ID:", target_ensembl_id)

Target name: EGFR
Target full name: Epidermal growth factor receptor
Target ChEMBL ID: CHEMBL203
Target Ensembl ID: ENSG00000146648


### 5. Helper functions

In [5]:
def normalize_name(value):
    """Normalise drug names for safer matching (uppercase, strip non-alphanumerics)."""
    if pd.isna(value):
        return ""
    value = str(value).upper()
    return re.sub(r"[^A-Z0-9]+", "", value)


def graphql_post(url, query, variables=None, retries=3, pause=1):
    """POST to a GraphQL endpoint with simple retries."""
    payload = {"query": query, "variables": variables or {}}
    for attempt in range(retries):
        try:
            response = requests.post(url, json=payload, timeout=(10, 60))
            if response.status_code == 200:
                return response.json()
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}, retrying...")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}, retrying...")
        time.sleep(pause)
    return None

### 6. Prepare project drug names

In [6]:
if "drug_name" not in drug_recommendations_df.columns:
    raise ValueError("drug_name column not found in egfr_drug_recommendations.csv")

project_drugs_df = (
    drug_recommendations_df[["drug_name"]].dropna().drop_duplicates().copy()
)
project_drugs_df["normalised_drug_name"] = project_drugs_df["drug_name"].apply(normalize_name)
project_drug_names = set(project_drugs_df["normalised_drug_name"])

print("Number of project drugs:", len(project_drugs_df))
project_drugs_df.head(20)

Number of project drugs: 76


,drug_name,normalised_drug_name
0,PANITUMUMAB,PANITUMUMAB
1,CETUXIMAB,CETUXIMAB
2,ERLOTINIB HYDROCHLORIDE,ERLOTINIBHYDROCHLORIDE
3,GEFITINIB,GEFITINIB
4,LAPATINIB DITOSYLATE,LAPATINIBDITOSYLATE
5,AFATINIB DIMALEATE,AFATINIBDIMALEATE
6,OSIMERTINIB MESYLATE,OSIMERTINIBMESYLATE
7,NECITUMUMAB,NECITUMUMAB
8,OLMUTINIB,OLMUTINIB
9,BRIGATINIB,BRIGATINIB


### 7. Query DGIdb for EGFR drug-gene interactions
Note: in the current DGIdb schema, `interactions` is a direct list (no `nodes` wrapper).

In [7]:
DGIDB_GRAPHQL_URL = "https://dgidb.org/api/graphql"

dgidb_query = """query interactionsByGene($geneNames: [String!]!) {
  genes(names: $geneNames) {
    nodes {
      name
      conceptId
      interactions {
        interactionScore
        drug { name conceptId }
        interactionTypes { type directionality }
        sources { sourceDbName }
      }
    }
  }
}"""

dgidb_response = graphql_post(
    url=DGIDB_GRAPHQL_URL,
    query=dgidb_query,
    variables={"geneNames": [target_name]},
)

if dgidb_response is None:
    print("DGIdb request failed.")
elif "errors" in dgidb_response:
    print("DGIdb returned GraphQL errors.")
    print(dgidb_response["errors"][:2])
else:
    print("DGIdb response received.")
    print(dgidb_response.keys())

DGIdb response received.
dict_keys(['data'])


### 8. Convert DGIdb response into a dataframe

In [8]:
dgidb_records = []

try:
    gene_nodes = (dgidb_response or {}).get("data", {}).get("genes", {}).get("nodes", [])
    for gene in gene_nodes:
        gene_name = gene.get("name")
        gene_concept_id = gene.get("conceptId")
        for interaction in gene.get("interactions", []) or []:
            drug = interaction.get("drug", {}) or {}
            itypes = interaction.get("interactionTypes", []) or []
            sources = interaction.get("sources", []) or []
            dgidb_records.append({
                "target_name": target_name,
                "gene_name": gene_name,
                "gene_concept_id": gene_concept_id,
                "drug_name": drug.get("name"),
                "drug_concept_id": drug.get("conceptId"),
                "interaction_score": interaction.get("interactionScore"),
                "interaction_types": " | ".join(i.get("type", "") for i in itypes if i.get("type")),
                "sources": " | ".join(s.get("sourceDbName", "") for s in sources if s.get("sourceDbName")),
                "source": "DGIdb",
            })
except Exception as error:
    print("Could not parse DGIdb response:", error)

dgidb_interactions_df = pd.DataFrame(dgidb_records)
if dgidb_interactions_df.empty:
    dgidb_interactions_df = pd.DataFrame(columns=[
        "target_name", "gene_name", "gene_concept_id", "drug_name", "drug_concept_id",
        "interaction_score", "interaction_types", "sources", "source"])

print("DGIdb interaction records:", len(dgidb_interactions_df))
dgidb_interactions_df.head(20)

DGIdb interaction records: 192


,target_name,gene_name,gene_concept_id,drug_name,drug_concept_id,interaction_score,interaction_types,sources,source
0,EGFR,EGFR,hgnc:3236,ILORASERTIB,ncit:C116729,0.002344,,DTC,DGIdb
1,EGFR,EGFR,hgnc:3236,ERLOTINIB,rxcui:337525,0.568066,inhibitor,ClearityFoundationClinicalTrial | TEND | CGI |...,DGIdb
2,EGFR,EGFR,hgnc:3236,MDX-447,chembl:CHEMBL2109391,0.135947,modulator,ChEMBL,DGIdb
3,EGFR,EGFR,hgnc:3236,ZALUTUMUMAB,ncit:C64620,0.815684,inhibitor,TTD | TALC | ChEMBL,DGIdb
4,EGFR,EGFR,hgnc:3236,RUSERONTINIB,ncit:C131492,0.090632,inhibitor,ChEMBL,DGIdb
5,EGFR,EGFR,hgnc:3236,LAPATINIB,rxcui:480167,0.271895,inhibitor,ClearityFoundationClinicalTrial | TEND | CGI |...,DGIdb
6,EGFR,EGFR,hgnc:3236,DACOMITINIB ANHYDROUS,rxcui:2058848,0.589105,inhibitor,TTD | MyCancerGenomeClinicalTrial | MyCancerGe...,DGIdb
7,EGFR,EGFR,hgnc:3236,AFATINIB,rxcui:1430438,1.223527,inhibitor,CGI | TTD | MyCancerGenomeClinicalTrial | MyCa...,DGIdb
8,EGFR,EGFR,hgnc:3236,RAMUCIRUMAB,rxcui:1535922,0.135947,,CIViC | PharmGKB | FDA,DGIdb
9,EGFR,EGFR,hgnc:3236,SIROLIMUS,rxcui:35302,0.028127,,CGI | CIViC | DoCM,DGIdb


### 9. Match DGIdb drugs to our project drug list

In [9]:
if not dgidb_interactions_df.empty:
    dgidb_interactions_df["normalised_drug_name"] = dgidb_interactions_df["drug_name"].apply(normalize_name)
    dgidb_interactions_df["matches_project_drug"] = dgidb_interactions_df["normalised_drug_name"].isin(project_drug_names)
else:
    dgidb_interactions_df["normalised_drug_name"] = pd.Series(dtype="object")
    dgidb_interactions_df["matches_project_drug"] = pd.Series(dtype="bool")

matched_dgidb_df = dgidb_interactions_df[dgidb_interactions_df["matches_project_drug"] == True].copy()
print("DGIdb records matching project drugs:", len(matched_dgidb_df))
matched_dgidb_df.head(20)

DGIdb records matching project drugs: 72


,target_name,gene_name,gene_concept_id,drug_name,drug_concept_id,interaction_score,interaction_types,sources,source,normalised_drug_name,matches_project_drug
2,EGFR,EGFR,hgnc:3236,MDX-447,chembl:CHEMBL2109391,0.135947,modulator,ChEMBL,DGIdb,MDX447,True
3,EGFR,EGFR,hgnc:3236,ZALUTUMUMAB,ncit:C64620,0.815684,inhibitor,TTD | TALC | ChEMBL,DGIdb,ZALUTUMUMAB,True
4,EGFR,EGFR,hgnc:3236,RUSERONTINIB,ncit:C131492,0.090632,inhibitor,ChEMBL,DGIdb,RUSERONTINIB,True
6,EGFR,EGFR,hgnc:3236,DACOMITINIB ANHYDROUS,rxcui:2058848,0.589105,inhibitor,TTD | MyCancerGenomeClinicalTrial | MyCancerGe...,DGIdb,DACOMITINIBANHYDROUS,True
10,EGFR,EGFR,hgnc:3236,DULIGOTUZUMAB,ncit:C116628,0.271895,antibody | inhibitor,TTD | MyCancerGenome | ChEMBL,DGIdb,DULIGOTUZUMAB,True
12,EGFR,EGFR,hgnc:3236,OSIMERTINIB,rxcui:1721560,0.155368,inhibitor,CGI | TTD | CIViC | PharmGKB | OncoKB | ChEMBL...,DGIdb,OSIMERTINIB,True
13,EGFR,EGFR,hgnc:3236,CETUXIMAB,rxcui:318341,0.617275,antibody | inhibitor,TEND | CGI | TTD | MyCancerGenome | CIViC | Do...,DGIdb,CETUXIMAB,True
14,EGFR,EGFR,hgnc:3236,ROCILETINIB,ncit:C99905,0.407842,inhibitor,CGI | TTD | MyCancerGenome | CIViC | TALC | Ch...,DGIdb,ROCILETINIB,True
18,EGFR,EGFR,hgnc:3236,FALNIDAMOL,ncit:C80868,0.108758,inhibitor,TTD | ChEMBL,DGIdb,FALNIDAMOL,True
19,EGFR,EGFR,hgnc:3236,BRIGATINIB,rxcui:1921217,0.074153,inhibitor,MyCancerGenomeClinicalTrial | CIViC | ChEMBL,DGIdb,BRIGATINIB,True


### 10. Save DGIdb interactions

In [10]:
dgidb_interactions_file = PROCESSED_DIR / "egfr_dgidb_interactions.csv"
dgidb_interactions_df.to_csv(dgidb_interactions_file, index=False)
print("Saved:", dgidb_interactions_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_dgidb_interactions.csv


### 11. Query Open Targets for EGFR disease associations

In [11]:
OPENTARGETS_GRAPHQL_URL = "https://api.platform.opentargets.org/api/v4/graphql"

opentargets_query = """query targetAssociatedDiseases($ensemblId: String!, $size: Int!) {
  target(ensemblId: $ensemblId) {
    id
    approvedSymbol
    approvedName
    associatedDiseases(page: { index: 0, size: $size }) {
      count
      rows { score disease { id name } }
    }
  }
}"""

opentargets_response = graphql_post(
    url=OPENTARGETS_GRAPHQL_URL,
    query=opentargets_query,
    variables={"ensemblId": target_ensembl_id, "size": 20},
)

if opentargets_response is None:
    raise RuntimeError("Open Targets request failed.")
if "errors" in opentargets_response:
    raise RuntimeError(opentargets_response["errors"])

print("Open Targets response received.")
print(opentargets_response.keys())

Open Targets response received.
dict_keys(['data'])


### 12. Convert Open Targets response into a dataframe

In [12]:
target_data = (opentargets_response or {}).get("data", {}).get("target", {}) or {}
associated_diseases = target_data.get("associatedDiseases", {}) or {}
association_rows = associated_diseases.get("rows", []) or []

opentargets_records = []
for row in association_rows:
    disease = row.get("disease", {}) or {}
    opentargets_records.append({
        "target_name": target_name,
        "target_full_name": target_data.get("approvedName"),
        "target_ensembl_id": target_data.get("id"),
        "approved_symbol": target_data.get("approvedSymbol"),
        "disease_id": disease.get("id"),
        "disease_name": disease.get("name"),
        "association_score": row.get("score"),
        "source": "Open Targets",
    })

opentargets_associations_df = pd.DataFrame(opentargets_records)
print("Total associated diseases (Open Targets):", associated_diseases.get("count"))
print("Rows fetched:", len(opentargets_associations_df))
opentargets_associations_df.head(20)

Total associated diseases (Open Targets): 2539
Rows fetched: 20


,target_name,target_full_name,target_ensembl_id,approved_symbol,disease_id,disease_name,association_score,source
0,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0003060,non-small cell lung carcinoma,0.852433,Open Targets
1,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,MONDO_0008903,lung cancer,0.766258,Open Targets
2,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0000571,lung adenocarcinoma,0.763945,Open Targets
3,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,MONDO_0004992,cancer,0.745873,Open Targets
4,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,MONDO_0007254,breast cancer,0.678243,Open Targets
5,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0000365,colorectal adenocarcinoma,0.669844,Open Targets
6,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0000181,head and neck squamous cell carcinoma,0.668905,Open Targets
7,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0006859,head and neck malignant neoplasia,0.654672,Open Targets
8,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0000616,neoplasm,0.654535,Open Targets
9,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0000519,glioblastoma multiforme,0.648555,Open Targets


### 13. Save Open Targets associations

In [13]:
opentargets_associations_file = PROCESSED_DIR / "egfr_opentargets_associations.csv"
opentargets_associations_df.to_csv(opentargets_associations_file, index=False)
print("Saved:", opentargets_associations_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_opentargets_associations.csv


### 14. Build the external cross-check summary

In [14]:
dgidb_interaction_count = len(dgidb_interactions_df)
matched_project_drug_count = len(matched_dgidb_df)
opentargets_associated_disease_count = associated_diseases.get("count", len(opentargets_associations_df))

if not opentargets_associations_df.empty:
    top_opentargets_diseases = " | ".join(
        opentargets_associations_df.sort_values("association_score", ascending=False)["disease_name"].dropna().head(5).tolist()
    )
else:
    top_opentargets_diseases = ""

external_crosscheck_summary_df = pd.DataFrame([{
    "target_name": target_name,
    "target_full_name": target_full_name,
    "target_chembl_id": target_chembl_id,
    "target_ensembl_id": target_ensembl_id,
    "dgidb_interaction_count": dgidb_interaction_count,
    "matched_project_drug_count": matched_project_drug_count,
    "opentargets_associated_disease_count": opentargets_associated_disease_count,
    "top_opentargets_diseases": top_opentargets_diseases,
}])
external_crosscheck_summary_df

,target_name,target_full_name,target_chembl_id,target_ensembl_id,dgidb_interaction_count,matched_project_drug_count,opentargets_associated_disease_count,top_opentargets_diseases
0,EGFR,Epidermal growth factor receptor,CHEMBL203,ENSG00000146648,192,72,2539,non-small cell lung carcinoma | lung cancer | ...


### 15. Add an external evidence score

In [15]:
def calculate_external_crosscheck_score(row):
    score = 0.0
    if row["dgidb_interaction_count"] > 0:
        score += 0.3
    if row["matched_project_drug_count"] > 0:
        score += 0.3
    if row["opentargets_associated_disease_count"] > 0:
        score += 0.3
    if row["opentargets_associated_disease_count"] >= 10:
        score += 0.1
    return round(min(score, 1.0), 2)


external_crosscheck_summary_df["external_crosscheck_score"] = external_crosscheck_summary_df.apply(
    calculate_external_crosscheck_score, axis=1
)
external_crosscheck_summary_df

,target_name,target_full_name,target_chembl_id,target_ensembl_id,dgidb_interaction_count,matched_project_drug_count,opentargets_associated_disease_count,top_opentargets_diseases,external_crosscheck_score
0,EGFR,Epidermal growth factor receptor,CHEMBL203,ENSG00000146648,192,72,2539,non-small cell lung carcinoma | lung cancer | ...,1.0


### 16. Save external cross-check summary

In [16]:
external_summary_file = PROCESSED_DIR / "egfr_external_crosscheck_summary.csv"
external_crosscheck_summary_df.to_csv(external_summary_file, index=False)
print("Saved:", external_summary_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_external_crosscheck_summary.csv


### 17. Final result

In [17]:
print("DGIdb and Open Targets Cross-Check Complete")
print("=" * 70)
print("Target:", target_name)
print("DGIdb interaction records:", len(dgidb_interactions_df))
print("DGIdb records matching project drugs:", len(matched_dgidb_df))
print("Open Targets disease rows fetched:", len(opentargets_associations_df))
print("Open Targets total associated disease count:", associated_diseases.get("count"))
display(external_crosscheck_summary_df)
display(dgidb_interactions_df.head(10))
display(opentargets_associations_df.head(10))

DGIdb and Open Targets Cross-Check Complete
Target: EGFR
DGIdb interaction records: 192
DGIdb records matching project drugs: 72
Open Targets disease rows fetched: 20
Open Targets total associated disease count: 2539


,target_name,target_full_name,target_chembl_id,target_ensembl_id,dgidb_interaction_count,matched_project_drug_count,opentargets_associated_disease_count,top_opentargets_diseases,external_crosscheck_score
0,EGFR,Epidermal growth factor receptor,CHEMBL203,ENSG00000146648,192,72,2539,non-small cell lung carcinoma | lung cancer | ...,1.0


,target_name,gene_name,gene_concept_id,drug_name,drug_concept_id,interaction_score,interaction_types,sources,source,normalised_drug_name,matches_project_drug
0,EGFR,EGFR,hgnc:3236,ILORASERTIB,ncit:C116729,0.002344,,DTC,DGIdb,ILORASERTIB,False
1,EGFR,EGFR,hgnc:3236,ERLOTINIB,rxcui:337525,0.568066,inhibitor,ClearityFoundationClinicalTrial | TEND | CGI |...,DGIdb,ERLOTINIB,False
2,EGFR,EGFR,hgnc:3236,MDX-447,chembl:CHEMBL2109391,0.135947,modulator,ChEMBL,DGIdb,MDX447,True
3,EGFR,EGFR,hgnc:3236,ZALUTUMUMAB,ncit:C64620,0.815684,inhibitor,TTD | TALC | ChEMBL,DGIdb,ZALUTUMUMAB,True
4,EGFR,EGFR,hgnc:3236,RUSERONTINIB,ncit:C131492,0.090632,inhibitor,ChEMBL,DGIdb,RUSERONTINIB,True
5,EGFR,EGFR,hgnc:3236,LAPATINIB,rxcui:480167,0.271895,inhibitor,ClearityFoundationClinicalTrial | TEND | CGI |...,DGIdb,LAPATINIB,False
6,EGFR,EGFR,hgnc:3236,DACOMITINIB ANHYDROUS,rxcui:2058848,0.589105,inhibitor,TTD | MyCancerGenomeClinicalTrial | MyCancerGe...,DGIdb,DACOMITINIBANHYDROUS,True
7,EGFR,EGFR,hgnc:3236,AFATINIB,rxcui:1430438,1.223527,inhibitor,CGI | TTD | MyCancerGenomeClinicalTrial | MyCa...,DGIdb,AFATINIB,False
8,EGFR,EGFR,hgnc:3236,RAMUCIRUMAB,rxcui:1535922,0.135947,,CIViC | PharmGKB | FDA,DGIdb,RAMUCIRUMAB,False
9,EGFR,EGFR,hgnc:3236,SIROLIMUS,rxcui:35302,0.028127,,CGI | CIViC | DoCM,DGIdb,SIROLIMUS,False


,target_name,target_full_name,target_ensembl_id,approved_symbol,disease_id,disease_name,association_score,source
0,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0003060,non-small cell lung carcinoma,0.852433,Open Targets
1,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,MONDO_0008903,lung cancer,0.766258,Open Targets
2,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0000571,lung adenocarcinoma,0.763945,Open Targets
3,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,MONDO_0004992,cancer,0.745873,Open Targets
4,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,MONDO_0007254,breast cancer,0.678243,Open Targets
5,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0000365,colorectal adenocarcinoma,0.669844,Open Targets
6,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0000181,head and neck squamous cell carcinoma,0.668905,Open Targets
7,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0006859,head and neck malignant neoplasia,0.654672,Open Targets
8,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0000616,neoplasm,0.654535,Open Targets
9,EGFR,epidermal growth factor receptor,ENSG00000146648,EGFR,EFO_0000519,glioblastoma multiforme,0.648555,Open Targets
